# Social Media & Screen Time by Generation — Data Prep

Trims the two APA Stress in America SPSS files down to the questions about **stress, social media and screen time**, plus age/generation and weights, and stacks them into one respondent-level table.

| Survey | File | Waves kept | Why |
|---|---|---|---|
| Aug 2016, Jan 2017, Aug 2017, Aug 2018 | `ds-1_2007-2018.sav` | 10, 11, 12, 13 | 2016 & 2018 reports focused on technology / social media; 2018 adds 692 teens (15-17) |
| Aug 2024 | `ds15_2019-2024.sav` | 24 | Only 2019-24 wave with social media & screen-time questions |

Outputs (in `data/processed/`):
- `sm_generations.csv` — one row per respondent, harmonized column names
- `sm_variable_dictionary.csv` — what each column is, the original question text, value labels, years asked, and whether it is comparable to 2024

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyreadstat

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

DATA = Path("data")
OUT = DATA / "processed"
OUT.mkdir(exist_ok=True)

## Variable map

Each entry: `(column, 2016-18 variable, 2024 variable, topic, comparability)`.

- **same** — identical question in both periods, can be compared directly
- **similar** — same idea, different wording or format; compare with care
- **2016-18 only** / **2024 only** — no counterpart

In [2]:
VARS = [
    # --- Stress & wellbeing ---
    ("stress_level",                 "Q605",     "Q605",    "stress", "same"),
    ("mental_health_rating",         "Q2255",    "Q2255",   "stress", "same"),          # 2018 + 2024 only
    ("physical_health_rating",       "Q2250",    "Q2250",   "stress", "same"),
    ("stress_change_past_year",      "Q623",     None,      "stress", "2016-18 only"),
    ("tech_stress_level",            "Q9005_1",  None,      "stress", "2016-18 only"),
    ("stress_source_technology",     "Q625R12",  None,      "stress", "2016-18 only"),
    ("stress_source_sm_perception",  "Q716R15",  None,      "stress", "2016-18 only"),  # 2018 teens only
    ("stress_source_false_news",     None,       "Q3226R5", "stress", "2024 only"),
    ("stress_source_ai",             None,       "Q3226R4", "stress", "2024 only"),

    # --- Social media & screen use ---
    ("uses_social_media",            "Q9016",    None,      "use", "2016-18 only"),
    ("sm_check_freq",                None,       None,      "use", "2016-18 only"),  # built below from Q9035B_3 / Q9035AR3
    ("internet_hours_week",          "DMNETUSE", None,      "use", "2016-18 only"),
    ("internet_use_freq",            "TEENINTUSE", "Q720B", "use", "similar"),       # 2018 teens vs 2024 adults; 6 vs 5 answer options
    ("online_hours_listening",       "Q9040_1",  None,      "use", "2016-18 only"),
    ("online_hours_reading",         "Q9040_2",  None,      "use", "2016-18 only"),
    ("online_hours_gaming",          "Q9040_3",  None,      "use", "2016-18 only"),
    ("online_hours_watching",        "Q9040_4",  None,      "use", "2016-18 only"),
    ("online_hours_connecting",      "Q9040_5",  None,      "use", "2016-18 only"),
    ("sm_contact_family_freq",       "Q710A_6",  None,      "use", "2016-18 only"),
    ("sm_contact_friends_freq",      "Q715A_6",  None,      "use", "2016-18 only"),
    ("close_ties_via_sm",            None,       "Q3567R5", "use", "similar"),       # ~ sm_contact_friends_freq
    ("close_ties_in_person",         None,       "Q3567R1", "use", "2024 only"),
    ("connecting_online_change",     None,       "Q3595R2", "use", "2024 only"),
    ("connecting_in_person_change",  None,       "Q3595R1", "use", "2024 only"),
    ("copes_by_going_online",        "QD966_23", None,      "use", "2016-18 only"),
    ("copes_via_social_media",       "QD966_24", None,      "use", "similar"),       # ~ sm_helpful_destress (2024)
    ("copes_video_games",            "QD966_3",  None,      "use", "2016-18 only"),
    ("shared_political_views_sm",    "Q1090R11", "Q1090R5", "use", "same"),          # 2017, 2018, 2024

    # --- Opinions on social media (2016 / 2018 agree-disagree) ---
    ("sm_worry_health_effects",      "Q9020R1",  None, "sm_opinion", "2016-18 only"),
    ("sm_worry_child_health",        "Q9020R2",  None, "sm_opinion", "2016-18 only"),
    ("sm_manage_image",              "Q9020R3",  None, "sm_opinion", "2016-18 only"),
    ("sm_share_good_not_bad",        "Q9020R4",  None, "sm_opinion", "2016-18 only"),
    ("sm_not_genuine_connection",    "Q9020R5",  None, "sm_opinion", "2016-18 only"),
    ("sm_less_in_person",            "Q9020R6",  None, "sm_opinion", "2016-18 only"),
    ("sm_share_joys",                "Q9020R7",  None, "sm_opinion", "2016-18 only"),
    ("sm_no_social_life_without",    "Q9020R8",  None, "sm_opinion", "2016-18 only"),
    ("sm_enjoy_expressing",          "Q9020R9",  None, "sm_opinion", "2016-18 only"),
    ("sm_helped_identity",           "Q9020R10", None, "sm_opinion", "2016-18 only"),
    ("sm_political_discussions_stress", "Q9020R11", None, "sm_opinion", "similar"),  # ~ sm_politics_hurt_mental_health (2024)
    ("sm_people_overshare",          "Q9020R12", None, "sm_opinion", "2016-18 only"),
    ("sm_feel_supported",            "Q9020R13", None, "sm_opinion", "2016-18 only"),
    ("sm_feel_judged",               "Q9020R14", None, "sm_opinion", "2016-18 only"),
    ("sm_feel_bad_about_self",       "Q9020R15", None, "sm_opinion", "similar"),     # ~ sm_compare_self (2024)
    ("sm_important_social_life",     "Q9020R16", None, "sm_opinion", "2016-18 only"),

    # --- Opinions on social media / screens (2024) ---
    ("sm_helpful_destress",          None, "Q3611R1", "sm_opinion", "similar"),
    ("sm_mindless_scrolling",        None, "Q3611R2", "sm_opinion", "2024 only"),
    ("sm_less_empathy",              None, "Q3611R5", "sm_opinion", "2024 only"),
    ("sm_quit_negative_content",     None, "Q3601R1", "sm_opinion", "2024 only"),
    ("sm_unsure_what_to_believe",    None, "Q3601R2", "sm_opinion", "2024 only"),
    ("sm_deleted_accounts",          None, "Q3601R3", "sm_opinion", "2024 only"),
    ("sm_pressure_stay_connected",   None, "Q3601R4", "sm_opinion", "2024 only"),
    ("sm_unfollowed_to_avoid",       None, "Q3601R5", "sm_opinion", "2024 only"),
    ("sm_compare_self",              None, "Q3601R6", "sm_opinion", "similar"),
    ("sm_none_of_these",             None, "Q3601R98", "sm_opinion", "2024 only"),
    ("sm_not_a_user",                None, "Q3601R99", "sm_opinion", "2024 only"),
    ("sm_politics_hurt_mental_health", None, "Q3675R1", "sm_opinion", "similar"),
    ("sm_avoid_during_election",     None, "Q3675R4", "sm_opinion", "2024 only"),
    ("believed_false_info_online",   None, "Q3675R5", "sm_opinion", "2024 only"),
    ("screen_too_dependent_on_phone", None, "Q3600R1", "screen", "similar"),        # ~ attached_to_phone (2016)
    ("screen_lost_hours",            None, "Q3600R2", "screen", "2024 only"),
    ("screen_phone_to_avoid_people", None, "Q3600R3", "screen", "2024 only"),
    ("screen_distracted_in_person",  None, "Q3600R4", "screen", "2024 only"),
    ("screen_feel_ignored_by_phones", None, "Q3600R5", "screen", "2024 only"),
    ("screen_worry_children",        None, "Q3600R6", "screen", "2024 only"),
    ("screen_no_negative_impact",    None, "Q3600R98", "screen", "2024 only"),
    ("screen_no_screentime",         None, "Q3600R99", "screen", "2024 only"),

    # --- Opinions on technology / screens (2016) ---
    ("tech_good_outweighs_bad",      "Q9010",    None, "screen", "2016-18 only"),
    ("tech_comfort",                 "Q9025",    None, "screen", "2016-18 only"),
    ("tech_more_educated",           "Q9015_1",  None, "screen", "2016-18 only"),
    ("tech_info_overwhelming",       "Q9015_2",  None, "screen", "2016-18 only"),
    ("tech_conflict_at_home",        "Q9015_3",  None, "screen", "2016-18 only"),
    ("tech_family_disconnected",     "Q9015_4",  None, "screen", "2016-18 only"),
    ("tech_have_tools_healthy_use",  "Q9015_5",  None, "screen", "2016-18 only"),
    ("child_screen_time_battle",     "Q9015_6",  None, "screen", "2016-18 only"),
    ("child_attached_to_phone",      "Q9015_7",  None, "screen", "2016-18 only"),
    ("attached_to_phone",            "Q9015_8",  None, "screen", "similar"),
    ("model_healthy_tech_for_kids",  "Q9015_9",  None, "screen", "2016-18 only"),
    ("unplug_important",             "Q9015_10", None, "screen", "2016-18 only"),
    ("tech_needed_track_tasks",      "Q9015_11", None, "screen", "2016-18 only"),
    ("tech_needed_know_friends",     "Q9015_12", None, "screen", "2016-18 only"),
    ("child_uses_sm",                "Q9017",    None, "screen", "2016-18 only"),
    ("assess_own_tech_use_freq",     "Q9060",    None, "screen", "2016-18 only"),
    ("limits_screens_before_bed",    "Q9065M1",  None, "screen", "2016-18 only"),
    ("no_phone_in_bedroom",          "Q9065M3",  None, "screen", "2016-18 only"),
    ("no_phone_at_dinner",           "Q9065M5",  None, "screen", "2016-18 only"),
    ("sm_notifications_off",         "Q9065M7",  None, "screen", "2016-18 only"),
    ("digital_detox",                "Q9065M11", None, "screen", "2016-18 only"),
    ("sets_screen_time_limits",      "Q9065M12", None, "screen", "2016-18 only"),
    ("most_stressful_tech_aspect",   "Q9056X1",  None, "screen", "2016-18 only"),
    ("tech_stress_pressure_to_respond", "Q9056X2M1", None, "screen", "2016-18 only"),
    ("tech_stress_comparing_on_sm",  "Q9056X2M6", None, "screen", "similar"),     # ~ sm_compare_self (2024)
    ("tech_stress_presenting_self",  "Q9056X2M7", None, "screen", "2016-18 only"),
    ("tech_stress_sm_arguments",     "Q9056X2M8", None, "screen", "2016-18 only"),
    ("tech_stress_constant_checking", "Q9056X2M9", None, "screen", "2016-18 only"),
]

# Answer codes that mean "not sure / declined / not applicable" -> NaN
MISSING_CODES = {
    "Q605": [98, 99],
    "Q9035B_3": [6], "Q9035AR3": [6],  # 'Not applicable' (mostly non-users)
    **{v: [5] for v in [f"Q9020R{i}" for i in range(1, 17)] + [f"Q9015_{i}" for i in range(1, 13)] + ["Q716R15"]},
    "Q9017": [3],  # Don't know
}

DEMO_0718 = ["YEAR", "WAVE", "RECORD", "DMAGE", "Q268B", "NETRACEUS", "NETEDUUS", "NETHHINCUS", "Q3004", "WT1", "WT2"]
DEMO_24 = ["YEAR", "WAVE", "RECORD", "DMAGE", "FINGENR", "NETEDUUS", "DMHHINCUS", "Q3004", "HSAMPLE", "NWEIGHTWT_P151954_GENPOP18PLUS"]
SM_CHECK = ["Q9035B_3", "Q9035AR3"]

vars_0718 = [v[1] for v in VARS if v[1]]
vars_24 = [v[2] for v in VARS if v[2]]
len(VARS)

94

## Load (only the columns we need)

In [3]:
raw_0718, meta_0718 = pyreadstat.read_sav(DATA / "ds-1_2007-2018.sav", usecols=DEMO_0718 + SM_CHECK + vars_0718)
raw_0718 = raw_0718[raw_0718["WAVE"].isin([10, 11, 12, 13])]

raw_24, meta_24 = pyreadstat.read_sav(DATA / "ds15_2019-2024.sav", usecols=DEMO_24 + vars_24)
raw_24 = raw_24[raw_24["WAVE"] == 24]

print("2016-18:", raw_0718.shape, " 2024:", raw_24.shape)
raw_0718.groupby("WAVE").size()

2016-18: (12520, 78)  2024: (3936, 43)


WAVE
10.0    3511
11.0    1019
12.0    3440
13.0    4550
dtype: int64

## Harmonize

- **Weights:** adults 2016-18 → `WT1`; 2018 teens → `WT2` (teen 15-17 weight); 2024 → general population 18+ weight. Teens and adults are weighted to different populations, so don't pool them into one "all ages" number.
- **`sm_check_freq`:** in 2016-17 the "on a typical day" version (`Q9035AR3`) only went to ~2,000 mostly-employed people, so we use the full-sample version (`Q9035B_3`, "How often do you check…"). 2018 asked `Q9035AR3` of everyone. Same 1-5 scale (Never → Constantly).
- **Gender:** sex at birth (`Q268B`) in 2016-18; final gender (`FINGENR`) in 2024 — the closest available match.

In [4]:
def clean(raw, source_col_idx, extra_missing=MISSING_CODES):
    out = pd.DataFrame(index=raw.index)
    for v in VARS:
        src = v[source_col_idx]
        if src:
            out[v[0]] = raw[src].where(~raw[src].isin(extra_missing.get(src, [])))
    return out

# 2016-18
a = clean(raw_0718, 1)
sm_b = raw_0718["Q9035B_3"].where(~raw_0718["Q9035B_3"].isin([6]))
sm_ar = raw_0718["Q9035AR3"].where(~raw_0718["Q9035AR3"].isin([6]))
a["sm_check_freq"] = np.where(raw_0718["WAVE"] == 13, sm_ar, sm_b)
is_teen = raw_0718["DMAGE"] < 18
a = pd.concat([pd.DataFrame({
    "survey": "2007-2018 file",
    "wave": raw_0718["WAVE"].astype(int),
    "year": raw_0718["YEAR"].astype(int),
    "respondent_id": raw_0718["WAVE"].astype(int).astype(str) + "-" + raw_0718["RECORD"].astype(int).astype(str),
    "age": raw_0718["DMAGE"],
    "is_teen": is_teen,
    "gender": raw_0718["Q268B"].map({1: "Male", 2: "Female"}),
    "education": raw_0718["NETEDUUS"],
    "children_in_household": raw_0718["Q3004"],
    "weight": np.where(is_teen, raw_0718["WT2"], raw_0718["WT1"]),
}), a], axis=1)

# 2024
b = clean(raw_24, 2)
b = pd.concat([pd.DataFrame({
    "survey": "2019-2024 file",
    "wave": raw_24["WAVE"].astype(int),
    "year": raw_24["YEAR"].astype(int),
    "respondent_id": raw_24["WAVE"].astype(int).astype(str) + "-" + raw_24["RECORD"].astype(int).astype(str),
    "age": raw_24["DMAGE"],
    "is_teen": False,
    "gender": raw_24["FINGENR"].map({1: "Male", 2: "Female", 9: "Other/Prefer not to answer"}),
    "education": raw_24["NETEDUUS"],
    "children_in_household": raw_24["Q3004"],
    "weight": raw_24["NWEIGHTWT_P151954_GENPOP18PLUS"],
}), b], axis=1)

df = pd.concat([a, b], ignore_index=True)
df.shape

(16456, 104)

## Generation and age band

**Generation** uses fixed birth years (Pew definitions) so a cohort can be followed from 2016 to 2024. Birth year ≈ survey year − age. APA's own reports re-cut generations by age each year, so our numbers won't exactly match the published ones.

| Generation | Born | Age in 2016 | Age in 2018 | Age in 2024 |
|---|---|---|---|---|
| Gen Z | 1997-2012 | 18-19 | 15-21 | 18-27 |
| Millennials | 1981-1996 | 20-35 | 22-37 | 28-43 |
| Gen X | 1965-1980 | 36-51 | 38-53 | 44-59 |
| Boomers | 1946-1964 | 52-70 | 54-72 | 60-78 |
| Silent | ≤1945 | 71+ | 73+ | 79+ |

**Age band** is the alternative: compares people *at the same age* across years (e.g. 18-24 in 2016 vs 18-24 in 2024).

Note: age is top-coded at 80, and all 2018 teens are coded as 17 (they're 15-17).

In [5]:
df["birth_year"] = df["year"] - df["age"]
GEN_ORDER = ["Gen Z", "Millennials", "Gen X", "Boomers", "Silent"]
df["generation"] = pd.cut(
    df["birth_year"], bins=[0, 1945, 1964, 1980, 1996, 2012],
    labels=GEN_ORDER[::-1],
).cat.reorder_categories(GEN_ORDER, ordered=True)

AGE_ORDER = ["15-17", "18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
df["age_band"] = pd.cut(df["age"], bins=[0, 17, 24, 34, 44, 54, 64, 200], labels=AGE_ORDER)

df["survey"] = df["survey"].astype("category")
df["gender"] = df["gender"].astype("category")
front = ["survey", "wave", "year", "respondent_id", "age", "birth_year", "generation", "age_band",
         "is_teen", "gender", "education", "children_in_household", "weight"]
df = df[front + [c for c in df.columns if c not in front]]
df.head()

,survey,wave,year,respondent_id,age,birth_year,generation,age_band,is_teen,gender,education,children_in_household,weight,stress_level,mental_health_rating,physical_health_rating,stress_change_past_year,tech_stress_level,stress_source_technology,stress_source_sm_perception,uses_social_media,internet_hours_week,internet_use_freq,online_hours_listening,online_hours_reading,online_hours_gaming,online_hours_watching,online_hours_connecting,sm_contact_family_freq,sm_contact_friends_freq,copes_by_going_online,copes_via_social_media,copes_video_games,shared_political_views_sm,sm_worry_health_effects,sm_worry_child_health,sm_manage_image,sm_share_good_not_bad,sm_not_genuine_connection,sm_less_in_person,sm_share_joys,sm_no_social_life_without,sm_enjoy_expressing,sm_helped_identity,sm_political_discussions_stress,sm_people_overshare,sm_feel_supported,sm_feel_judged,sm_feel_bad_about_self,sm_important_social_life,tech_good_outweighs_bad,tech_comfort,tech_more_educated,tech_info_overwhelming,tech_conflict_at_home,tech_family_disconnected,tech_have_tools_healthy_use,child_screen_time_battle,child_attached_to_phone,attached_to_phone,model_healthy_tech_for_kids,unplug_important,tech_needed_track_tasks,tech_needed_know_friends,child_uses_sm,assess_own_tech_use_freq,limits_screens_before_bed,no_phone_in_bedroom,no_phone_at_dinner,sm_notifications_off,digital_detox,sets_screen_time_limits,most_stressful_tech_aspect,tech_stress_pressure_to_respond,tech_stress_comparing_on_sm,tech_stress_presenting_self,tech_stress_sm_arguments,tech_stress_constant_checking,sm_check_freq,stress_source_false_news,stress_source_ai,close_ties_via_sm,close_ties_in_person,connecting_online_change,connecting_in_person_change,sm_helpful_destress,sm_mindless_scrolling,sm_less_empathy,sm_quit_negative_content,sm_unsure_what_to_believe,sm_deleted_accounts,sm_pressure_stay_connected,sm_unfollowed_to_avoid,sm_compare_self,sm_none_of_these,sm_not_a_user,sm_politics_hurt_mental_health,sm_avoid_during_election,believed_false_info_online,screen_too_dependent_on_phone,screen_lost_hours,screen_phone_to_avoid_people,screen_distracted_in_person,screen_feel_ignored_by_phones,screen_worry_children,screen_no_negative_impact,screen_no_screentime
0,2007-2018 file,10,2016,10-764,63.0,1953.0,Boomers,55-64,False,Female,2.0,NaN,1.23455,NaN,NaN,5.0,2.0,1.0,1.0,NaN,1.0,3.0,NaN,2.0,2.0,2.0,1.0,2.0,1.0,1.0,0.0,0.0,0.0,NaN,1.0,NaN,1.0,2.0,2.0,1.0,4.0,1.0,2.0,2.0,1.0,3.0,NaN,NaN,NaN,NaN,2.0,4.0,3.0,2.0,1.0,1.0,3.0,NaN,NaN,3.0,NaN,2.0,3.0,3.0,NaN,1.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2007-2018 file,10,2016,10-767,35.0,1981.0,Millennials,35-44,False,Male,2.0,NaN,1.20387,1.0,NaN,3.0,2.0,1.0,1.0,NaN,1.0,5.0,NaN,2.0,4.0,1.0,4.0,3.0,1.0,6.0,1.0,0.0,0.0,NaN,1.0,NaN,2.0,4.0,2.0,1.0,4.0,1.0,3.0,1.0,1.0,3.0,NaN,NaN,NaN,NaN,2.0,4.0,4.0,2.0,1.0,1.0,4.0,NaN,NaN,3.0,NaN,2.0,1.0,3.0,NaN,5.0,0.0,0.0,0.0,0.0,0.0,0.0,13.0,0.0,0.0,0.0,0.0,1.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2007-2018 file,10,2016,10-768,55.0,1961.0,Boomers,55-64,False,Female,3.0,NaN,0.40091,3.0,NaN,2.0,3.0,1.0,1.0,NaN,1.0,4.0,NaN,1.0,3.0,2.0,1.0,2.0,6.0,1.0,1.0,0.0,0.0,NaN,1.0,NaN,1.0,3.0,2.0,1.0,3.0,1.0,3.0,1.0,3.0,3.0,NaN,NaN,NaN,NaN,2.0,4.0,3.0,2.0,1.0,2.0,3.0,NaN,NaN,3.0,NaN,3.0,1.0,1.0,NaN,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2007-2018 file,10,2016,10-769,80.0,1936.0,Silent,65+,False,Female,2.0,NaN,4.41872,4.0,NaN,3.0,3.0,5.0,1.0,NaN,1.0,4.0,NaN,1.0,1.0,2.0,4.0,2.0,6.0,6.0,0.0,0.0,0.0,NaN,1.0,NaN,3.0,3.0,3.0,1.0,4.0,1.0,3.0,1.0,2.0,4.0,NaN,NaN,NaN,NaN,2.0,3.0,2.0,3.0,1.0,1.0,2.0,NaN,NaN,1.0,NaN,1.0,1.0,1.0,NaN,1.0,0.0,0.0,0.0,0.0,0.0,0.0,15.0,0.0,0.0,0.0,1.0,0.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

## Checks

In [6]:
# Respondents per generation and year (teens split out)
pd.crosstab([df["generation"], df["is_teen"]], df["year"], margins=True)

year                 2016  2017  2018  2024    All
generation  is_teen                               
Gen Z       False     100   115   631   540   1386
            True        0     0   692     0    692
Millennials False    1333   997   936  1215   4481
Gen X       False    1181   922   854  1120   4077
Boomers     False    1562  1176  1213   980   4931
Silent      False     354   230   224    81    889
All                  4530  3440  4550  3936  16456

In [7]:
# Weights present for everyone, and out-of-range values
assert df["weight"].notna().all() and (df["weight"] > 0).all()
assert df["respondent_id"].is_unique
assert df["stress_level"].dropna().between(1, 10).all()
df.groupby("year")["weight"].agg(["count", "mean", "min", "max"]).round(2)

,count,mean,min,max
year,,,,
2016,4530,1.0,0.04,10.01
2017,3440,1.0,0.08,6.37
2018,4550,1.0,0.08,6.35
2024,3936,1.0,0.20,5.00


In [8]:
# Which questions were asked in which year (% of respondents with an answer)
coverage = (df.groupby("year")[[v[0] for v in VARS]].count().T / df.groupby("year").size() * 100).round(0)
coverage.columns = coverage.columns.astype(str)
coverage

year,2016,2017,2018,2024
stress_level,98.0,97.0,100.0,99.0
mental_health_rating,0.0,0.0,100.0,100.0
physical_health_rating,78.0,100.0,100.0,100.0
stress_change_past_year,78.0,100.0,100.0,0.0
tech_stress_level,77.0,0.0,0.0,0.0
...,...,...,...,...
tech_stress_pressure_to_respond,78.0,0.0,0.0,0.0
tech_stress_comparing_on_sm,78.0,0.0,0.0,0.0
tech_stress_presenting_self,78.0,0.0,0.0,0.0
tech_stress_sm_arguments,78.0,0.0,0.0,0.0


## Variable dictionary

In [9]:
def describe(var, meta):
    if not var:
        return "", ""
    labels = meta.variable_value_labels.get(var, {})
    return meta.column_names_to_labels.get(var, ""), json.dumps({int(k): v for k, v in labels.items()})

rows = []
for col, v0718, v24, topic, comparability in VARS:
    q0718, labels0718 = describe(v0718, meta_0718)
    q24, labels24 = describe(v24, meta_24)
    if col == "sm_check_freq":
        v0718 = "Q9035B_3 (2016-17) / Q9035AR3 (2018)"
        q0718, labels0718 = describe("Q9035AR3", meta_0718)
    years = coverage.columns[coverage.loc[col] > 0].tolist()
    rows.append({
        "column": col, "topic": topic, "comparability": comparability,
        "years_asked": ", ".join(years),
        "var_2016_18": v0718 or "", "var_2024": v24 or "",
        "question_2016_18": q0718, "question_2024": q24,
        "value_labels_2016_18": labels0718, "value_labels_2024": labels24,
    })
dictionary = pd.DataFrame(rows)
dictionary[["column", "topic", "comparability", "years_asked", "var_2016_18", "var_2024"]]

,column,topic,comparability,years_asked,var_2016_18,var_2024
0,stress_level,stress,same,"2016, 2017, 2018, 2024",Q605,Q605
1,mental_health_rating,stress,same,"2018, 2024",Q2255,Q2255
2,physical_health_rating,stress,same,"2016, 2017, 2018, 2024",Q2250,Q2250
3,stress_change_past_year,stress,2016-18 only,"2016, 2017, 2018",Q623,
4,tech_stress_level,stress,2016-18 only,2016,Q9005_1,
...,...,...,...,...,...,...
89,tech_stress_pressure_to_respond,screen,2016-18 only,2016,Q9056X2M1,
90,tech_stress_comparing_on_sm,screen,similar,2016,Q9056X2M6,
91,tech_stress_presenting_self,screen,2016-18 only,2016,Q9056X2M7,
92,tech_stress_sm_arguments,screen,2016-18 only,2016,Q9056X2M8,


In [10]:
# Pairs that can be compared 2016-18 -> 2024
dictionary[dictionary["comparability"].isin(["same", "similar"])][["column", "comparability", "years_asked", "question_2016_18", "question_2024"]]

,column,comparability,years_asked,question_2016_18,question_2024
0,stress_level,same,"2016, 2017, 2018, 2024","On a scale of 1 to 10, where 1 means you have ""little or no stress"" and 10 means you have ""a gre...","Q605 Next we have some questions about stress.On a scale of 1 to 10, where 1 means you have ''li..."
1,mental_health_rating,same,"2018, 2024","In general, would you say your mental health is ...","Q2255 In general, would you say your mental health is ..."
2,physical_health_rating,same,"2016, 2017, 2018, 2024","In general, would you say your physical health is ...","Q2250 In general, would you say your physical health is ..."
12,internet_use_freq,similar,"2018, 2024","Overall, how often do you use the Internet?",Q720B About how often do you use the internet?
20,close_ties_via_sm,similar,2024,,"Q3567r5: Through social media (e.g., Facebook, Twitter/X, Instagram, etc.) - How do you most com..."
25,copes_via_social_media,similar,"2016, 2017, 2018",Do you do any of the following to manage stress?: Sound off on social media,
27,shared_political_views_sm,same,"2017, 2018, 2024","Which of the following actions, if any, have you taken in the past year?: Shared or expressed my...",Q1090r5: Shared or expressed my political or social views on social media - Which of the followi...
38,sm_political_discussions_stress,similar,"2016, 2018",Please indicate the extent to which you agree or disagree with the following statements as it sp...,
42,sm_feel_bad_about_self,similar,2018,Please indicate the extent to which you agree or disagree with the following statements as it sp...,
44,sm_helpful_destress,similar,2024,,Q3611r1: I find that social media can be a helpful way to de-stress. - How much do you agree or ...


## Save

In [11]:
df.to_csv(OUT / "sm_generations.csv", index=False)
dictionary.to_csv(OUT / "sm_variable_dictionary.csv", index=False)
print(f"Saved {df.shape[0]:,} respondents x {df.shape[1]} columns")

Saved 16,456 respondents x 107 columns


## Quick look: same-question comparisons by generation

Weighted %, adults only. Agree items = somewhat/strongly agree.

In [12]:
def wpct(g, col, test):
    g = g[g[col].notna()]
    return np.average(test(g[col]), weights=g["weight"]) * 100 if len(g) >= 50 else np.nan

adults = df[~df["is_teen"]]

def summarize(g):
    stress = g[g["stress_level"].notna()]
    return pd.Series({
        "Avg stress (1-10)": np.average(stress["stress_level"], weights=stress["weight"]),
        "Extreme stress 8-10 (%)": wpct(g, "stress_level", lambda s: s >= 8),
        "Mental health fair/poor (%)": wpct(g, "mental_health_rating", lambda s: s <= 2),
        "Shared political views on SM (%)": wpct(g, "shared_political_views_sm", lambda s: s == 1),
        "n": len(g),
    })

cols = ["stress_level", "mental_health_rating", "shared_political_views_sm", "weight"]
summary = adults.groupby(["year", "generation"], observed=True)[cols].apply(summarize).round(1)
summary

Avg stress (1-10)  Extreme stress 8-10 (%)  \
year generation                                                
2016 Gen Z                      5.3                     21.0   
     Millennials                5.7                     27.0   
     Gen X                      5.4                     23.1   
     Boomers                    4.3                     17.6   
     Silent                     3.0                      7.8   
2017 Gen Z                      5.1                     17.2   
     Millennials                5.9                     30.9   
     Gen X                      5.3                     20.8   
     Boomers                    3.9                     12.3   
     Silent                     3.3                      7.8   
2018 Gen Z                      5.5                     23.4   
     Millennials                5.7                     23.8   
     Gen X                      5.3                     21.0   
     Boomers                    4.1                     11.0   
     Silent                     3.3                     12.6   
2024 Gen Z                      6.0                     30.8   
     Millennials                5.8                     26.4   
     Gen X                      5.1                     22.5   
     Boomers                    3.9                     12.3   
     Silent                     3.1                      2.9   

                  Mental health fair/poor (%)  \
year generation                                 
2016 Gen Z                                NaN   
     Millennials                          NaN   
     Gen X                                NaN   
     Boomers                              NaN   
     Silent                               NaN   
2017 Gen Z                                NaN   
     Millennials                          NaN   
     Gen X                                NaN   
     Boomers                              NaN   
     Silent                               NaN   
2018 Gen Z                               31.7   
     Millennials                         14.8   
     Gen X                               13.3   
     Boomers                              6.7   
     Silent                               5.0   
2024 Gen Z                               38.6   
     Millennials                         26.9   
     Gen X                               20.0   
     Boomers                              8.8   
     Silent                               5.4   

                  Shared political views on SM (%)       n  
year generation                                             
2016 Gen Z                                     NaN   100.0  
     Millennials                               NaN  1333.0  
     Gen X                                     NaN  1181.0  
     Boomers                                   NaN  1562.0  
     Silent                                    NaN   354.0  
2017 Gen Z                                    18.2   115.0  
     Millennials                              21.4   997.0  
     Gen X                                    18.6   922.0  
     Boomers                                  16.1  1176.0  
     Silent                                   15.7   230.0  
2018 Gen Z                                    22.1   631.0  
     Millennials                              20.5   936.0  
     Gen X                                    14.6   854.0  
     Boomers                                  15.7  1213.0  
     Silent                                   12.9   224.0  
2024 Gen Z                                    31.7   540.0  
     Millennials                              26.6  1215.0  
     Gen X                                    21.2  1120.0  
     Boomers                                  17.2   980.0  
     Silent                                    4.5    81.0